# Nowcast

Predicting Canadian gross domestic product (GDP, chained 2017 dollars,
seasonally adjusted at annual rates) before official publication using
employment, consumer price index, and manufacturing sales as leading
indicators.

**Models compared:**
- OLS Regression (baseline)
- Ridge Regression (L2-regularised linear model)
- Support Vector Regression (SVR) with RBF kernel
- Gradient Boosting Regressor
- Random Forest Regressor (primary model)

## 1. Load Data

Read the feature-engineered dataset produced by `transform.py`.
The file contains 21 lag / rolling-mean / growth-rate features plus the
GDP target (`gdp_value`).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings("ignore")

df = pd.read_csv("data/processed/training_data.csv", parse_dates=["date"])
print(f"Rows:        {df.shape[0]:,}")
print(f"Columns:     {df.shape[1]}")
print(f"Date range:  {df['date'].min().date()}  to  {df['date'].max().date()}")
print(f"GDP missing: {df['gdp_value'].isna().sum()}  (nowcast rows)")

## 2. Train / Nowcast Split

Rows where `gdp_value` is published go into the training set.
The latest row (where GDP may not be published yet) is the **nowcast target**.

In [ ]:
target = "gdp_value"
feature_cols = [c for c in df.columns if c not in ["date", target]]

train = df[df[target].notna()].copy()
X = train[feature_cols]
y = train[target]

latest = df.iloc[[-1]]
X_nowcast = latest[feature_cols]
last_date = latest["date"].iloc[0]
actual = latest[target].iloc[0]
actual_val = float(actual) if not pd.isna(actual) else None

print(f"Training rows:   {len(train)}")
print(f"Features:        {len(feature_cols)}")
print(f"Nowcast date:    {last_date.date()}")
if actual_val is not None:
    print(f"Actual GDP:      {actual_val:,.0f}")
else:
    print("Actual GDP:      Not yet published")

## 3. Helper: TimeSeries Cross-Validation

A reusable function that evaluates any model with 4-fold TimeSeries CV
and returns the average MAE and MAE%.

In [ ]:
tscv = TimeSeriesSplit(n_splits=4)

def cv_mae(model, X, y):
    scores = []
    for tr, te in tscv.split(X):
        m = model.__class__(**model.get_params())
        m.fit(X.iloc[tr], y.iloc[tr])
        preds = m.predict(X.iloc[te])
        scores.append(mean_absolute_error(y.iloc[te], preds))
    avg = float(np.mean(scores))
    pct = 100 * avg / float(y.mean())
    return avg, pct

## 4. OLS Regression (Baseline)

Simple linear model with no regularisation. Serves as a reference point.

In [ ]:
ols = LinearRegression()
ols_cv_mae, ols_cv_pct = cv_mae(ols, X, y)
ols.fit(X, y)
ols_pred = float(ols.predict(X_nowcast)[0])

print("OLS Regression")
print(f"  Nowcast:  {ols_pred:,.0f}")
if actual_val is not None:
    err = abs(ols_pred - actual_val)
    print(f"  Error:    {err:,.0f}  ({100 * err / actual_val:.2f}%)")
print(f"  CV MAE:   {ols_cv_mae:,.1f}  ({ols_cv_pct:.2f}%)")

## 5. Ridge Regression (L2 Regularisation)

Linear model with L2 penalty — handles multicollinearity among the
21 lag / rolling / growth features better than plain OLS.

In [ ]:
ridge = Ridge(alpha=1.0, random_state=42)
ridge_cv_mae, ridge_cv_pct = cv_mae(ridge, X, y)
ridge.fit(X, y)
ridge_pred = float(ridge.predict(X_nowcast)[0])

print("Ridge Regression")
print(f"  Nowcast:  {ridge_pred:,.0f}")
if actual_val is not None:
    err = abs(ridge_pred - actual_val)
    print(f"  Error:    {err:,.0f}  ({100 * err / actual_val:.2f}%)")
print(f"  CV MAE:   {ridge_cv_mae:,.1f}  ({ridge_cv_pct:.2f}%)")

## 6. Support Vector Regression (SVR)

SVR with an RBF kernel captures non-linear relationships.
Features are standardised internally for SVR's benefit.

In [ ]:
# SVR benefits from scaled features
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)
X_nowcast_scaled = pd.DataFrame(scaler.transform(X_nowcast), columns=X_nowcast.columns, index=X_nowcast.index)

svr = SVR(kernel="rbf", C=100, gamma="scale")
svr_cv_mae, svr_cv_pct = cv_mae(svr, X_scaled, y)
svr.fit(X_scaled, y)
svr_pred = float(svr.predict(X_nowcast_scaled)[0])

print("Support Vector Regression (RBF)")
print(f"  Nowcast:  {svr_pred:,.0f}")
if actual_val is not None:
    err = abs(svr_pred - actual_val)
    print(f"  Error:    {err:,.0f}  ({100 * err / actual_val:.2f}%)")
print(f"  CV MAE:   {svr_cv_mae:,.1f}  ({svr_cv_pct:.2f}%)")

## 7. Gradient Boosting Regressor

Sequential ensemble that builds trees one after another, each correcting
the errors of the previous. Often outperforms Random Forest on structured data.

In [ ]:
gbr = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gbr_cv_mae, gbr_cv_pct = cv_mae(gbr, X, y)
gbr.fit(X, y)
gbr_pred = float(gbr.predict(X_nowcast)[0])

print("Gradient Boosting Regressor")
print(f"  Nowcast:  {gbr_pred:,.0f}")
if actual_val is not None:
    err = abs(gbr_pred - actual_val)
    print(f"  Error:    {err:,.0f}  ({100 * err / actual_val:.2f}%)")
print(f"  CV MAE:   {gbr_cv_mae:,.1f}  ({gbr_cv_pct:.2f}%)")

## 8. Random Forest Regressor

Bagging ensemble of 100 decision trees. The primary model for the project.
Evaluated with 4-fold TimeSeries CV then refit on all training data.

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf_cv_mae, rf_cv_pct = cv_mae(rf, X, y)
rf.fit(X, y)
rf_pred = float(rf.predict(X_nowcast)[0])

print("Random Forest Regressor")
print(f"  Nowcast:  {rf_pred:,.0f}")
if actual_val is not None:
    err = abs(rf_pred - actual_val)
    print(f"  Error:    {err:,.0f}  ({100 * err / actual_val:.2f}%)")
print(f"  CV MAE:   {rf_cv_mae:,.1f}  ({rf_cv_pct:.2f}%)")

## 9. Model Comparison

Side-by-side comparison of all 5 models on the nowcast target.

In [ ]:
results = []
models = [
    ("OLS Regression",          ols_pred,  ols_cv_mae,  ols_cv_pct),
    ("Ridge Regression",        ridge_pred, ridge_cv_mae, ridge_cv_pct),
    ("SVR (RBF)",               svr_pred,   svr_cv_mae,   svr_cv_pct),
    ("Gradient Boosting",       gbr_pred,   gbr_cv_mae,   gbr_cv_pct),
    ("Random Forest",           rf_pred,    rf_cv_mae,    rf_cv_pct),
]

print(f"{'Model':<22} {'Nowcast':>12} {'Error':>12} {'MAE%':>8}  {'CV MAE':>10} {'CV MAE%':>8}")
print("-" * 74)

best_model = min(models, key=lambda m: m[2])  # lowest CV MAE

for name, pred, cv_mae_val, cv_pct_val in models:
    err = abs(pred - actual_val) if actual_val is not None else float("nan")
    err_pct = 100 * err / actual_val if actual_val is not None else float("nan")
    marker = "  ← best" if name == best_model[0] else ""
    print(f"{name:<22} {pred:>12,.0f} {err:>12,.0f} {err_pct:>7.2f}%  {cv_mae_val:>10,.0f} {cv_pct_val:>7.2f}%{marker}")

if actual_val is None:
    print("\n⚠ Actual GDP not yet published — all nowcasts are forward predictions.")

## 10. Feature Importance (Random Forest)

Which features drive the Random Forest the most?  Employment levels
(`lfs_value`) typically dominate, followed by CPI rolling means.

In [ ]:
imp = pd.DataFrame({"feature": feature_cols, "importance": rf.feature_importances_})
imp = imp.sort_values("importance", ascending=False).reset_index(drop=True)

print("Top 10 features by importance:")
print(f"{'':>3} {'Feature':<28} {'Importance':>10}")
print("-" * 43)
for i, row in imp.head(10).iterrows():
    print(f"{i+1:>2}.  {row['feature']:<28} {row['importance']:.6f}")

print("\nImportance by predictor group:")
for prefix, label in [("lfs", "Employment"), ("cpi", "CPI"), ("mfg", "Manufacturing")]:
    group_imp = imp[imp["feature"].str.startswith(prefix)]["importance"].sum()
    print(f"  {label:<25} {group_imp:.4f}")

---
*Notebook generated on data from `data/processed/training_data.csv`.*